In [12]:
import pickle
import datasets
import pandas as pd

original_dataset = pd.read_csv("../.dataset/dabstep_data_default.csv")
with open("../results/opus_dabstep_test_no_skills_150examples.pkl", 'rb') as f:
    data = pickle.load(f)

In [13]:
answers = []

for result in data:
    question = result.question
    answer = result.trace.output.final_answer
    reasoning = result.trace.output.reasoning
    tid = str(result.index)
    task_answer = {
        "task_id": tid,
        "agent_answer": str(answer),
        "reasoning_trace": str(reasoning),
    }
    answers.append(task_answer)



In [14]:
# Get task_ids we actually have results for
processed_task_ids = {str(a["task_id"]) for a in answers}

# Find task_ids in original_dataset that weren't processed
original_task_ids = original_dataset["task_id"].astype(str).unique()
missing_task_ids = set(original_task_ids) - processed_task_ids

# Build a lookup for question by task_id
task_to_question = original_dataset.set_index(
    original_dataset["task_id"].astype(str)
)["question"].to_dict()

In [15]:
for tid in missing_task_ids:
    answers.append({
        "task_id": tid,
        "agent_answer": None,
        "reasoning_trace": None,
    })

In [16]:
results_df = pd.DataFrame(answers)

In [17]:
from pathlib import Path

# Crate dir to store agent run results
RUNS_DIR = Path().resolve() / "runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

def write_jsonl(data: list[dict], filepath: Path) -> None:
    """Write a list of dictionaries to a JSONL file."""
    # Ensure the directory exists
    filepath.parent.mkdir(parents=True, exist_ok=True)

    with open(filepath, "w") as file:
        for entry in data:
            file.write(json.dumps(entry) + "\n")

In [18]:
RUNS_DIR

PosixPath('/Users/salahalzubi/cursor_projects/deep_agents/notebooks/runs')

In [19]:
import json
write_jsonl(answers, RUNS_DIR / "opus_dabstep_test_no_skills_150examples.jsonl")

In [ ]:
submission_id = "Sentient-sentient-skills-150examples"


In [11]:
from datasets import load_dataset
import itertools

submission_id = "Sentient-sentient-skills-150examples"

# Open the table in streaming mode – rows are decoded on‑the‑fly
stream = load_dataset(
    "adyen/DABstep",
    data_dir="task_scores",
    split="default",
    streaming=True
)

# Pull rows that match the id, stop after we have all 450 matches
# (the generator will automatically stop when the stream ends)
matches = list(itertools.islice(
    (r for r in stream if r["submission_id"] == submission_id),
    5000          # a safe upper bound – the generator stops early at 450
))

Using the latest cached version of the dataset since adyen/DABstep couldn't be found on the Hugging Face Hub


ValueError: Couldn't find cache for adyen/DABstep for config 'default-data_dir=task_scores'
Available configs in the cache: ['tasks']